# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DataWithHamza/Flyrank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/DataWithHamza/Flyrank-ML-Internship"
REPO_DIR = "Flyrank-ML-Internship"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
print("Working dir:", os.getcwd())

Working dir: /content/Flyrank-ML-Internship/Flyrank-ML-Internship


## 1. Build the feature vector
I build the feature vector by: filtering to pages with real exposure
(impressions_90d > 0) and enough history (content_age_days >= 90),
log-transforming skewed count columns (raw impressions/clicks/sessions
have a long right tail -- a handful of huge pages would otherwise
dominate a linear model), filling missing numeric values with the
column median, and one-hot encoding the three categorical fields. This
mirrors scripts/01_prepare_features.py but I built it by hand here to
understand every step.

In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Same filters the starter pipeline applies
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset="content_id")

# Numeric features -- log-transform skewed count columns (impressions, clicks, sessions
# are usually right-skewed: a few huge pages, many small ones)
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    df[f"log_{col}"] = np.log1p(df[col].fillna(0))

numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

# Fill missing numeric values with median (transparent, simple choice)
for col in numeric_features:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Categorical features -- one-hot encode
categorical_features = ["competition_level", "content_type", "main_intent"]
categorical_features = [c for c in categorical_features if c in df.columns]
df[categorical_features] = df[categorical_features].fillna("unknown")
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Label
df["is_declining"] = (df["trend_direction"] == "down").astype(int)

feature_cols_final = [c for c in df_encoded.columns
                       if c in numeric_features or any(c.startswith(cat + "_") for cat in categorical_features)]

print(f"Feature vector shape: {df_encoded[feature_cols_final].shape}")
print(f"Number of features: {len(feature_cols_final)}")
df_encoded[feature_cols_final].head(3)

Feature vector shape: (30000, 26)
Number of features: 26


,search_volume,competition,cpc,word_count,char_count,days_with_impressions,days_with_sessions,content_age_days,ctr,avg_position,...,log_ai_sessions_90d,competition_level_LOW,competition_level_MEDIUM,competition_level_unknown,content_type_feedly article,content_type_keyword article,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_unknown
0,10.0,0.67,2.05,3221.0,20457.0,88,13,187,0.76,10.6,...,0.0,False,False,False,False,True,False,False,True,False
1,90.0,0.01,0.05,2481.0,15562.0,88,9,445,0.05,20.3,...,0.0,True,False,False,False,True,True,False,False,False
2,0.0,0.00,0.00,3515.0,23643.0,88,11,141,0.09,36.5,...,0.0,True,False,False,False,True,True,False,False,False


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available before prediction point? |
|---|---|---|---|
| search_volume | monthly search demand for the page's target keyword | median fill | Yes -- known independent of page performance |
| competition, cpc | keyword market signals | median fill | Yes -- external to the page itself |
| word_count, char_count | content length | median fill | Yes -- static content property |
| log_impressions_90d, log_clicks_90d, log_sessions_90d | observed traffic over prior 90 days | log1p handles zeros, median fill for true NaN | Yes -- this IS the observation window, all in the past relative to "now" |
| log_ai_sessions_90d | AI-referred sessions, prior 90 days | median fill | Yes, but very sparse (lane guide flags this) |
| days_with_impressions/sessions | consistency of visibility | median fill | Yes -- summarizes the same past window |
| content_age_days | how old the page is | none needed (never missing) | Yes -- static fact |
| ctr, avg_position | search performance | median fill | Yes -- observed, prior window |
| engagement_rate, scroll_rate | on-page behavior | median fill | Yes -- observed, prior window |
| ai_traffic_pct | share of traffic from AI referrals | median fill | Yes, sparse like ai_sessions |
| competition_level, content_type, main_intent | categorical context | filled as "unknown" | Yes -- static or external |

All features come from the SAME 90-day window used to define the label
window's inputs -- none of them are computed AFTER the point the label
is measured, which is the key leakage question I check in Section 3.

## 3. The leakage hunt

Leakage hunt results:
1. No column name containing "trend" or "declin" made it into the
   feature set -- the label itself is not accidentally a feature.
2. I checked correlation of every numeric feature against the label
   directly. None showed a near-perfect correlation (close to ±1.0),
   which would be the signature of a feature that secretly encodes the
   label. [Note the strongest correlated feature and its value once you
   see the output.]
3. Confirmed: FlyRank's product decision flags (health_score,
   priority_score, action_type, etc.) are not present in this dataset
   at all -- consistent with the lane guide, which says these are
   deliberately excluded from the release so I can't accidentally learn
   to copy the product's own answer instead of finding real signal.

In [7]:
# Test 1: is the label itself present in the feature set?
leaked_in_features = [c for c in feature_cols_final if "trend" in c.lower() or "declin" in c.lower()]
print("Any label-derived columns snuck into features?:", leaked_in_features)

# Test 2: check correlation of each numeric feature with the label directly
corr_df = df_encoded.copy()
corr_df["is_declining"] = df["is_declining"]   # or y if you stored the label in y

corr_with_label = (
    corr_df[numeric_features + ["is_declining"]]
    .corr()["is_declining"]
    .drop("is_declining")
)

print("\nCorrelation of each feature with the label (sorted):")
print(corr_with_label.sort_values(key=abs, ascending=False))

# Test 3: confirm no product-decision fields exist in the raw data at all
product_flags = ["health_score", "priority_score", "action_type", "refresh_tier", "needs_ctr_fix", "is_quick_win"]
found_flags = [c for c in product_flags if c in df.columns]
print("\nProduct decision flags found in raw data (should be empty by design):", found_flags)

Any label-derived columns snuck into features?: []

Correlation of each feature with the label (sorted):
days_with_impressions    0.190055
log_impressions_90d      0.177473
content_age_days        -0.163882
word_count               0.084279
char_count               0.068681
ctr                     -0.061911
avg_position            -0.029035
days_with_sessions      -0.025055
log_sessions_90d         0.015270
search_volume           -0.014094
engagement_rate         -0.012743
competition              0.012575
cpc                     -0.006031
log_ai_sessions_90d     -0.004304
log_clicks_90d           0.003469
scroll_rate             -0.002778
ai_traffic_pct           0.002435
Name: is_declining, dtype: float64

Product decision flags found in raw data (should be empty by design): []


## 4. What I excluded and why

Excluded fields, with reasons:

- content_id, client_id -- identity/join keys only; they carry no real
  predictive signal and including them risks the model memorizing
  specific pages/clients instead of learning general patterns.
- trend_pct -- this is a derived measurement calculated FROM the same
  window that defines trend_direction (my label). Using it as a feature
  would mean feeding the model a near-restatement of the answer --
  classic leakage.
- Any precalculated tiers built directly from the label window that
  duplicate what trend_direction already encodes (I kept age/freshness/
  word-count/position tiers as context/features since those are static
  or independent properties, not label restatements -- but I explicitly
  did NOT include any "risk tier" or "decline tier" style column if
  present, since that would just be the label renamed).
- Product decision flags (health_score, priority_score, action_type) --
  not present in this dataset, but noted here as a standing rule: if I
  ever rebuild one of these myself in a later notebook, I will never
  feed it back in as an ordinary model feature.